# Reusable Decision Tree Classification Workflow with Python

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mojiahui777/ba_course/blob/main/decision_tree_credit_risk_demo.ipynb)

> **Student goal:** Replace the example CSV with another classification dataset and complete the entire workflow.
>
> 1. Select **File → Save a copy in Drive**.
> 2. Run the example once with `Credit.csv`.
> 3. Upload your own CSV and edit only the **dataset settings** cell in Section 4.
> 4. Run **Runtime → Run all**, then interpret and discuss the new results.

## Learning objectives

By the end of this activity, you will be able to:

- define the target and predictor roles for a new dataset;
- inspect data quality and class balance;
- prepare numeric and categorical predictors;
- train and visualize a decision tree;
- evaluate it against a simple baseline; and
- explain the limitations of the result.

## 1. Define the business problem

Before coding, write down:

- **Observation:** What does one row represent?
- **Target:** What category are you trying to predict?
- **Predictors:** What information is available before the prediction?
- **Decision:** How would someone use the prediction?
- **Error costs:** Are the two types of classification error equally serious?

**Example:** In `Credit.csv`, one row represents an applicant and `CREDIT` is the recorded class. The file does not verify whether `0` or `1` means Good, so keep the numeric labels unless a matching data dictionary supplies that meaning.

## 2. Import the Python libraries

In [ ]:
import io
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree

## 3. Load a CSV

- Keep `UPLOAD_FILE = False` to run the provided example from GitHub.
- Change it to `True` to upload a different CSV in Google Colab.

The rest of the notebook uses the DataFrame named `df`, regardless of which CSV you choose.

In [ ]:
EXAMPLE_URL = "https://raw.githubusercontent.com/mojiahui777/ba_course/main/Credit.csv"
UPLOAD_FILE = False  # Students: change to True to upload another CSV

if UPLOAD_FILE:
    from google.colab import files

    uploaded = files.upload()
    uploaded_name, uploaded_bytes = next(iter(uploaded.items()))
    df = pd.read_csv(io.BytesIO(uploaded_bytes))
    print(f"Loaded uploaded file: {uploaded_name}")
else:
    try:
        df = pd.read_csv(EXAMPLE_URL)
        print("Loaded the Credit.csv example from GitHub")
    except Exception:
        local_path = Path("Credit.csv")
        if not local_path.exists():
            raise FileNotFoundError(
                "Could not load the example. Upload a CSV or place Credit.csv beside the notebook."
            )
        df = pd.read_csv(local_path)
        print("Loaded local Credit.csv")

print(f"Dataset shape: {df.shape[0]} rows × {df.shape[1]} columns")
df.head()

## 4. Configure your dataset — edit this section

This is the bridge between **your CSV** and the reusable workflow. Every dataset uses different column names, so the model must know each column's role.

Students normally edit only these settings:

1. `TARGET_COLUMN`: the category to predict;
2. `ID_COLUMNS`: identifiers to exclude, or `[]` if there are none;
3. `CATEGORICAL_COLUMNS`: categorical predictors, including numeric category codes; and
4. optional model settings such as test size and tree depth.

The values below are the defaults for `Credit.csv`. Replace them after uploading another dataset.

In [ ]:
# ---------- STUDENTS: EDIT THIS SETTINGS CELL FOR YOUR DATA ----------
TARGET_COLUMN = "CREDIT"
ID_COLUMNS = ["OBS#"]
CATEGORICAL_COLUMNS = [
    "CHK_ACCT", "JOB", "NUM_DEPENDENTS", "TELEPHONE", "FOREIGN"
]

TEST_SIZE = 0.20
RANDOM_STATE = 42
MAX_DEPTH = 3
# --------------------------------------------------------------------

### Why is Section 4 necessary?

Python can detect text columns, but it cannot reliably guess the business role of every field:

- an ID may look numeric but should not predict the target;
- a category code such as `0, 1, 2, 3` may look numeric but is not a measured quantity; and
- only you know which column is the outcome of interest.

The checks below give clear messages when the settings do not match the uploaded CSV.

In [ ]:
required_columns = [TARGET_COLUMN] + ID_COLUMNS + CATEGORICAL_COLUMNS
missing_columns = [column for column in required_columns if column not in df.columns]
if missing_columns:
    raise ValueError(
        f"These configured columns are not in the CSV: {missing_columns}. "
        f"Available columns: {list(df.columns)}"
    )

if TARGET_COLUMN in ID_COLUMNS or TARGET_COLUMN in CATEGORICAL_COLUMNS:
    raise ValueError("TARGET_COLUMN must not also appear in ID_COLUMNS or CATEGORICAL_COLUMNS.")

print("Configuration accepted.")

## 5. Understand and inspect the data

Do not skip this step when changing datasets. Confirm what one row means, whether the target is sensible, and whether types and missing values match your expectations.

In [ ]:
display(df.head())
print("Column data types:")
display(df.dtypes.rename("dtype").to_frame())
print("Summary:")
display(df.describe(include="all").round(2))

## 6. Check data quality and target balance

Rows with a missing target cannot teach a supervised model, so they are removed. Missing predictor values will be imputed later inside the preprocessing pipeline.

A **majority-class baseline** predicts the most frequent class every time. It is a minimum comparison point—not a sufficient definition of a good model.

In [ ]:
missing_target_rows = df[TARGET_COLUMN].isna().sum()
if missing_target_rows:
    print(f"Removing {missing_target_rows} rows with a missing target.")
    df = df.dropna(subset=[TARGET_COLUMN]).copy()

print("Missing predictor values:")
display(df.drop(columns=[TARGET_COLUMN]).isna().sum().rename("missing").to_frame())

target_counts = df[TARGET_COLUMN].value_counts(dropna=False)
display(target_counts.rename("count").to_frame())
print(f"Majority-class baseline accuracy: {target_counts.max() / target_counts.sum():.3f}")

target_counts.plot(kind="bar", color="#CC0000", rot=0)
plt.xlabel(TARGET_COLUMN)
plt.ylabel("Number of observations")
plt.title("Target Class Distribution")
plt.tight_layout()
plt.show()

## 7. Select predictors and build preprocessing

The workflow:

- excludes the target and ID columns;
- one-hot encodes categorical predictors;
- fills missing categorical values with the most frequent category;
- fills missing numeric values with the median; and
- leaves numeric values unscaled because decision trees do not require scaling.

In [ ]:
feature_columns = [
    column for column in df.columns
    if column not in ID_COLUMNS + [TARGET_COLUMN]
]
categorical_columns = [
    column for column in CATEGORICAL_COLUMNS if column in feature_columns
]
numeric_columns = [
    column for column in feature_columns if column not in categorical_columns
]

if not feature_columns:
    raise ValueError("No predictors remain after removing the target and ID columns.")

X = df[feature_columns].copy()
y = df[TARGET_COLUMN].copy()

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
])

transformers = []
if categorical_columns:
    transformers.append(("category", categorical_pipeline, categorical_columns))
if numeric_columns:
    transformers.append(("number", numeric_pipeline, numeric_columns))

preprocessor = ColumnTransformer(
    transformers=transformers,
    verbose_feature_names_out=False,
)

print("Target:", TARGET_COLUMN)
print("Excluded IDs:", ID_COLUMNS)
print("Categorical predictors:", categorical_columns)
print("Numeric predictors:", numeric_columns)

## 8. Create a train-test split

The test set estimates performance on observations not used to fit the tree. Stratification is used when each class has enough rows; otherwise, the notebook prints a warning and performs an ordinary random split.

In [ ]:
class_counts = y.value_counts()
can_stratify = len(class_counts) > 1 and class_counts.min() >= 2
stratify_target = y if can_stratify else None
if not can_stratify:
    print("Warning: too few rows in at least one class for stratification.")

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=stratify_target,
)

print(f"Training rows: {len(X_train)}")
print(f"Testing rows:  {len(X_test)}")
display(y_train.value_counts().rename("training count").to_frame())
display(y_test.value_counts().rename("testing count").to_frame())

## 9. Train a decision tree

`criterion="entropy"` selects splits using information gain. `MAX_DEPTH` limits complexity so the tree is easier to read and less likely to memorize a small training sample.

Scikit-learn's tree is a teaching analogue—not an exact implementation of Weka J48/C4.5.

In [ ]:
model = Pipeline(steps=[
    ("preprocess", preprocessor),
    (
        "tree",
        DecisionTreeClassifier(
            criterion="entropy",
            max_depth=MAX_DEPTH,
            random_state=RANDOM_STATE,
        ),
    ),
])

model.fit(X_train, y_train)
print("Model fitted.")

## 10. Evaluate holdout predictions

- **Accuracy** is the overall correct share.
- **Balanced accuracy** gives equal weight to recall for each class.
- **Precision, recall, and F1** show class-specific performance.

Always compare the tree with the majority baseline.

In [ ]:
y_pred = model.predict(X_test)
baseline_accuracy = y_train.value_counts(normalize=True).max()

print(f"Test accuracy:              {accuracy_score(y_test, y_pred):.3f}")
print(f"Test balanced accuracy:     {balanced_accuracy_score(y_test, y_pred):.3f}")
print(f"Training majority baseline: {baseline_accuracy:.3f}")
print("\nPer-class report:")
print(classification_report(y_test, y_pred, zero_division=0))

## 11. Read the confusion matrix

Rows are actual classes and columns are predicted classes. Off-diagonal cells are errors. Translate those errors into business meaning using your dataset's verified class definitions.

In [ ]:
class_labels = list(model.named_steps["tree"].classes_)
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    labels=class_labels,
    display_labels=[str(value) for value in class_labels],
    cmap="Reds",
    colorbar=False,
)
plt.title("Decision Tree: Holdout Confusion Matrix")
plt.xlabel("Predicted class")
plt.ylabel("Actual class")
plt.tight_layout()
plt.show()

## 12. Visualize and read the tree

Start at the root. Move left when a condition is true and right when it is false. Each node reports entropy, sample count, class counts, and the predicted class.

In [ ]:
tree_model = model.named_steps["tree"]
feature_names = model.named_steps["preprocess"].get_feature_names_out()

plt.figure(figsize=(18, 9))
plot_tree(
    tree_model,
    feature_names=feature_names,
    class_names=[str(value) for value in tree_model.classes_],
    filled=True,
    rounded=True,
    fontsize=8,
)
plt.title(f"Entropy-Based Decision Tree (Maximum Depth {MAX_DEPTH})")
plt.tight_layout()
plt.show()

## 13. Translate the tree into text rules

In [ ]:
print(export_text(tree_model, feature_names=list(feature_names)))

## 14. Explore feature importance

Importance summarizes impurity reduction in this fitted tree. It is not proof of causality, fairness, or stability.

In [ ]:
importance = pd.Series(
    tree_model.feature_importances_, index=feature_names
).sort_values(ascending=False)
used_features = importance[importance > 0]

display(used_features.round(3).rename("importance").to_frame())
if used_features.empty:
    print("The fitted tree did not use any predictor. Check the data and settings.")
else:
    used_features.sort_values().plot(kind="barh", figsize=(8, 4), color="#CC0000")
    plt.xlabel("Impurity-based importance")
    plt.ylabel("Transformed feature")
    plt.title("Features Used by the Fitted Tree")
    plt.tight_layout()
    plt.show()

## 15. Optional experiment: class weighting

If the target is imbalanced, a class-weighted tree gives mistakes on less frequent classes more influence. Compare the errors; do not assume weighting automatically makes the model better.

In [ ]:
weighted_model = Pipeline(steps=[
    ("preprocess", preprocessor),
    (
        "tree",
        DecisionTreeClassifier(
            criterion="entropy",
            max_depth=MAX_DEPTH,
            class_weight="balanced",
            random_state=RANDOM_STATE,
        ),
    ),
])
weighted_model.fit(X_train, y_train)
weighted_pred = weighted_model.predict(X_test)

comparison = pd.DataFrame(
    {
        "Model": ["Original tree", "Class-weighted tree"],
        "Accuracy": [
            accuracy_score(y_test, y_pred),
            accuracy_score(y_test, weighted_pred),
        ],
        "Balanced accuracy": [
            balanced_accuracy_score(y_test, y_pred),
            balanced_accuracy_score(y_test, weighted_pred),
        ],
    }
).set_index("Model")
display(comparison.round(3))

## 16. Student submission checklist

For your new dataset, submit:

1. the business question and meaning of one row;
2. the edited settings cell;
3. data-quality and target-balance observations;
4. the train/test sizes and baseline;
5. the fitted tree and two example rules;
6. accuracy, balanced accuracy, and confusion matrix interpretation;
7. the most important features, with a non-causal interpretation; and
8. at least three limitations or risks.

**Takeaway:** The code can be reused, but the meaning, validity, and consequences must be reconsidered for every new dataset.